In [9]:
import os

from dotenv import load_dotenv
from Bio import Entrez, SeqIO
from Bio.Seq import Seq
from cb.seq.translation import check_residue, translate_cds

load_dotenv()

# 1. Retrieve sequences from NCBI and debug
Entrez.email = os.environ["NCBI_EMAIL"]
Entrez.api_key = os.getenv("NCBI_API_KEY")
handle = Entrez.efetch(db="nucleotide", id="NM_000518", rettype="gb", retmode="text")
record = SeqIO.read(handle, "genbank")
print(f"Retrieved record: {record.id} - {record.description}")

# 2. Translate the coding region (CDS) to protein
# The GenBank record gives the CDS location, so no hard-coded slice is needed.
protein_sequence = translate_cds(record)
print(f"Protein sequence: {protein_sequence}")

# 3. Identify the variant
# HBB uses legacy mature-protein numbering: the initiator Met (index 0) is
# cleaved off, so residue 6 = index 6. HGVS counts the Met, so it calls this
# same variant p.Glu7Val.
# check_residue raises if index 6 is neither E nor V, so a wrong offset fails loudly.
if check_residue(protein_sequence, 6, ref="E", alt="V") == "variant":
    print("Sickle cell variant detected: Glu6Val")
else:
    print("Healthy hemoglobin detected")

# 4. Positive control: exercise the Val branch
# Substitute the codon for residue 6 (GAG -> GTG, the HbS mutation) in a copy of the real CDS.
cds = next(f for f in record.features if f.type == "CDS")
cds_seq = cds.extract(record.seq)
codon = str(cds_seq[18:21])
assert codon == "GAG", f"unexpected codon {codon} at residue 6"
mutant_cds = cds_seq[:18] + Seq("GTG") + cds_seq[21:]
mutant_protein = mutant_cds.translate(to_stop=True)
print(f"Mutant protein:   {mutant_protein}")

if check_residue(mutant_protein, 6, ref="E", alt="V") == "variant":
    print("Sickle cell variant detected: Glu6Val")
else:
    print("Healthy hemoglobin detected")

Retrieved record: NM_000518.5 - Homo sapiens hemoglobin subunit beta (HBB), mRNA
Protein sequence: MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH
Healthy hemoglobin detected
Mutant protein:   MVHLTPVEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH
Sickle cell variant detected: Glu6Val
